# Figure 2 — pretreatment peripheral immune programmes

Run cells from top to bottom. Outputs and resource tables are written only to `resources/Figure2/`. Figure 2a and 2d are manuscript artwork, intentionally retained as documented manual panels rather than reconstructed code.


In [ ]:
# Locate the package regardless of whether Jupyter was started from the project root or notebooks/.
candidates <- c(
  getwd(),
  file.path(getwd(), ".."),
  file.path(getwd(), "figure_2_3_reproduction"),
  file.path(getwd(), "work_path", "figure_2_3_reproduction")
)
package_dir <- candidates[file.exists(file.path(candidates, "functions", "00_setup.R"))][1]
if (is.na(package_dir)) stop("Cannot locate figure_2_3_reproduction. Set package_dir manually.")
package_dir <- normalizePath(package_dir)
options(figure_package_dir = package_dir)
source(file.path(package_dir, "functions", "00_setup.R"))
check_figure_packages()
function_dir <- file.path(package_dir, "functions")

source(file.path(function_dir, '01_palettes.R'))
source(file.path(function_dir, '02_cell_frequency.R'))
source(file.path(function_dir, '03_clinical_timeline.R'))
source(file.path(function_dir, '04_program_definitions.R'))
source(file.path(function_dir, '05_program_scoring.R'))
source(file.path(function_dir, '06_survival.R'))

seurat_merge <- load_seurat_final()
meta <- seurat_merge@meta.data
clinical_df <- make_clinical_table(meta)
fig_dir <- make_resource_dir('Figure2')


## Fig. 2a — workflow

Manual artwork; the full verified panel is retained in `reference_docx_media/image2.png`. No analysis code was used to generate this schematic.


## Fig. 2b — patient sampling, response and 6-month PFS

Confirmed function: `plot_cns_clinical_matrix`.


In [ ]:
seurat_merge <- prepare_figure2b_metadata(seurat_merge)
p_fig2b <- plot_cns_clinical_matrix(seurat_merge)
print(p_fig2b)
save_ggplot_pdf(p_fig2b, 'Fig2b_clinical_sampling_matrix.pdf', 'Figure2', 12, 3)
fig2b_table <- seurat_merge@meta.data |> dplyr::select(patient_id, sample_type, response, pfs_6m) |> dplyr::distinct()
write_resource_table(fig2b_table, 'Fig2b_clinical_sampling_matrix.csv', 'Figure2')


## Fig. 2c — baseline cell-subset frequency (PFS >6 versus <6 months)

Confirmed core function and parameters: `calculate_cluster_diff(... group_rn, PFS>6_D0, PFS<6_D0, methods='part')`.


In [ ]:
diff_results_fig2c <- calculate_cluster_diff(seurat_merge, group_var = 'group_rn', cell_cluster = 'cell_type3', group1 = 'PFS>6_D0', group2 = 'PFS<6_D0', methods = 'part', paired = FALSE)
fig2c <- plot_cell_frequency_volcano(diff_results_fig2c, 'Fig. 2c: baseline immune-cell subset frequencies')
print(fig2c$plot)
save_ggplot_pdf(fig2c$plot, 'Fig2c_baseline_cell_frequency.pdf', 'Figure2', 6, 6)
write_resource_table(fig2c$data, 'Fig2c_baseline_cell_frequency.csv', 'Figure2')


## Fig. 2d — five-axis immune-program framework

Manual schematic in the manuscript. The program/subtype/gene resource table below is the computational source underlying the axes.


In [ ]:
fig2d_programs <- program_resource_table(immune_programs_targeted)
write_resource_table(fig2d_programs, 'Fig2d_immune_program_definitions.csv', 'Figure2')
fig2d_programs


## Fig. 2e — baseline (D0) patient-level immune-program heatmap

Confirmed score method: `AddModuleScore`; this notebook explicitly uses D0.


In [ ]:
score_res_fig2_d0 <- calculate_targeted_patient_scores(seurat_merge, immune_programs_targeted, patient_col = 'patient_id', timepoint_col = 'sample_type_rn', subtype_col = 'cell_type3', baseline_value = 'D0', assay = 'RNA', slot = 'data', min_cells_per_patient_program = 5, min_genes_present = 3, score_method = 'AddModuleScore')
patient_score_fig2_d0 <- score_res_fig2_d0$patient_program_matrix
ht_fig2e <- plot_patient_program_heatmap(patient_score_mat = patient_score_fig2_d0, clinical_df = clinical_df, patient_col = 'patient_id', order_by = 'pfs_time', scale_rows = TRUE, cluster_rows = TRUE, cluster_columns = TRUE, clinical_anno_cols = c('response', 'pfs_group'), output_pdf = file.path(fig_dir, 'Fig2e_baseline_program_heatmap.pdf'), width = 15, height = 12, title = 'Baseline patient-level immune programs')
write_resource_table(score_res_fig2_d0$patient_scores_long, 'Fig2e_D0_patient_program_scores.csv', 'Figure2')


## Fig. 2f — selected D0 programs by PFS group

The caption-selected programs are cDC1 cross-presentation, CD4 Th1 antiviral helper, CD8 cytotoxicity and CD8 chemokine trafficking.


In [ ]:
fig2f_scores <- score_res_fig2_d0$patient_scores_long |> dplyr::left_join(clinical_df |> dplyr::select(patient_id, pfs_group), by = 'patient_id')
fig2f_programs <- c('DC_cDC1__cross_presentation', 'CD4_all__Th1_antiviral_helper', 'CD8_all__cytotoxicity', 'CD8_all__chemokine_trafficking')
fig2f <- plot_program_boxplot_from_score_res(fig2f_scores, programs = fig2f_programs, group_col = 'pfs_group', group_values = c('PFS<6', 'PFS>6'), group_labels = c('PFS <6 months', 'PFS >6 months'), comparisons = list(c('PFS <6 months', 'PFS >6 months')), test_method = 'wilcox.test', alternative = 'two.sided', show_p_adj = FALSE, p_label_type = 'p', facet_ncol = 4, title = NULL, subtitle = NULL, y_label = 'Mean module score', x_label = 'PFS group', output_pdf = file.path(fig_dir, 'Fig2f_selected_baseline_programs.pdf'), width = 12, height = 4.5)
print(fig2f$plot)
write_resource_table(fig2f$plot_data, 'Fig2f_selected_baseline_program_scores.csv', 'Figure2')
write_resource_table(fig2f$stat_data, 'Fig2f_selected_baseline_program_statistics.csv', 'Figure2')


## Fig. 2g — baseline PFS associations

Confirmed endpoint: `pfs_status`. Confirmed CD4_LAG3 checkpoint-exhaustion call uses `AddModuleScore`.


In [ ]:
fig2g_cd8trm <- plot_stage_cell_survival(meta = meta, target_cell = 'CD8_TRM', target_stage = 'D0', cell_col = 'cell_type3', time_col = 'sample_type_rn', patient_col = 'patient_id', surv_time_col = 'pfs_time', surv_status_col = 'pfs_status')
save_survival_pdf(fig2g_cd8trm, 'Fig2g_CD8_TRM_PFS.pdf', 'Figure2')
write_resource_table(fig2g_cd8trm$data, 'Fig2g_CD8_TRM_PFS_resource.csv', 'Figure2')

fig2g_activation <- plot_geneset_pseudobulk_survival_v3(seurat_merge, gene_set = immune_programs_targeted$CD8_all__activation$genes, target_cell = immune_programs_targeted$CD8_all__activation$subtypes, target_stage = 'D0', score_name = 'CD8_Activation', surv_time_col = 'pfs_time', surv_status_col = 'pfs_status', score_method = 'AddModuleScore', risk_table_height = 0.25)
save_survival_pdf(fig2g_activation, 'Fig2g_CD8_activation_PFS.pdf', 'Figure2')
write_resource_table(fig2g_activation$data, 'Fig2g_CD8_activation_PFS_resource.csv', 'Figure2')

fig2g_lag3 <- plot_geneset_pseudobulk_survival_v3(seurat_merge, gene_set = immune_programs_targeted$CD4_LAG3__checkpoint_exhaustion$genes, target_cell = immune_programs_targeted$CD4_LAG3__checkpoint_exhaustion$subtypes, cell_col = 'cell_type3', target_stage = 'D0', score_name = 'Checkpoint_Exhaustion', surv_time_col = 'pfs_time', surv_status_col = 'pfs_status', score_method = 'AddModuleScore', risk_table_height = 0.25)
save_survival_pdf(fig2g_lag3, 'Fig2g_CD4_LAG3_checkpoint_exhaustion_PFS.pdf', 'Figure2')
write_resource_table(fig2g_lag3$data, 'Fig2g_CD4_LAG3_checkpoint_exhaustion_PFS_resource.csv', 'Figure2')
